# Qwen Inference

## Objective

This notebook performs inference using the Qwen model on a reasoning dataset.

### Workflow

1. Load the reasoning dataset.
2. Load the Qwen model.
3. Generate responses.
4. Save the generated responses.

In [9]:
!pip install -q transformers accelerate sentencepiece pandas tqdm

In [10]:
import os
import time

import torch
import pandas as pd

from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [4]:
print(torch.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

2.11.0+cu128
Device: cuda


In [7]:
!git clone https://github.com/MR-just01/open_vs_closed_LLM-.git

Cloning into 'open_vs_closed_LLM-'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 20 (delta 6), reused 17 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 4.61 KiB | 1.54 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [6]:
!ls

open_vs_closed_LLM-  sample_data


In [11]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:
def ask_qwen(question):

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start = time.time()

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    end = time.time()

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return answer, end-start

In [13]:
results = []

In [15]:
%cd open_vs_closed_LLM-
!git pull

/content/open_vs_closed_LLM-
Already up to date.


In [20]:
!ls data

chatgpt_results.csv  qwen_results.csv
final_scores.csv     reasoning_questions.csv


In [16]:
import pandas as pd

questions = pd.read_csv("data/reasoning_questions.csv")

questions.head()

,id,category,difficulty,reasoning_type,question,expected_answer
0,1,Logic,Easy,Deductive,If five machines make five toys in five minute...,5 minutes
1,2,Logic,Easy,Deductive,All roses are flowers. Some flowers fade quick...,No
2,3,Logic,Easy,Pattern,"Which number comes next: 2, 4, 8, 16, ?",32
3,4,Logic,Medium,Conditional,"If all A are B and all B are C, are all A C?",Yes
4,5,Logic,Medium,Deductive,John is taller than Mary. Mary is taller than ...,John


In [17]:
questions = pd.read_csv("data/reasoning_questions.csv")

In [18]:
results = []

In [19]:
for index, row in questions.iterrows():

    print(f"Processing Question {index + 1}/{len(questions)}")

    answer, runtime = ask_qwen(row["question"])

    results.append({
        "id": row["id"],
        "category": row["category"],
        "difficulty": row["difficulty"],
        "reasoning_type": row["reasoning_type"],
        "question": row["question"],
        "expected_answer": row["expected_answer"],
        "response": answer,
        "time_seconds": runtime
    })

Processing Question 1/50
Processing Question 2/50
Processing Question 3/50
Processing Question 4/50
Processing Question 5/50
Processing Question 6/50
Processing Question 7/50
Processing Question 8/50
Processing Question 9/50
Processing Question 10/50
Processing Question 11/50
Processing Question 12/50
Processing Question 13/50
Processing Question 14/50
Processing Question 15/50
Processing Question 16/50
Processing Question 17/50
Processing Question 18/50
Processing Question 19/50
Processing Question 20/50
Processing Question 21/50
Processing Question 22/50
Processing Question 23/50
Processing Question 24/50
Processing Question 25/50
Processing Question 26/50
Processing Question 27/50
Processing Question 28/50
Processing Question 29/50
Processing Question 30/50
Processing Question 31/50
Processing Question 32/50
Processing Question 33/50
Processing Question 34/50
Processing Question 35/50
Processing Question 36/50
Processing Question 37/50
Processing Question 38/50
Processing Question 3

In [20]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "qwen_results.csv",
    index=False
)

In [21]:
results_df.head()

,id,category,difficulty,reasoning_type,question,expected_answer,response,time_seconds
0,1,Logic,Easy,Deductive,If five machines make five toys in five minute...,5 minutes,"To solve this problem, let's break down the in...",11.909853
1,2,Logic,Easy,Deductive,All roses are flowers. Some flowers fade quick...,No,"No, we cannot conclude that all roses fade qui...",9.631236
2,3,Logic,Easy,Pattern,"Which number comes next: 2, 4, 8, 16, ?",32,The sequence you've provided is a series of po...,6.094239
3,4,Logic,Medium,Conditional,"If all A are B and all B are C, are all A C?",Yes,"To determine if all A are C given that ""all A ...",10.553561
4,5,Logic,Medium,Deductive,John is taller than Mary. Mary is taller than ...,John,Based on the information provided:\n\n1. John ...,2.892158


In [22]:
len(results_df)

50

In [23]:
results_df["response"].isnull().sum()

np.int64(0)

In [37]:
questions.head()

,id,category,difficulty,reasoning_type,question,expected_answer
0,1,Logic,Easy,Deductive,If five machines make five toys in five minute...,5 minutes
1,2,Logic,Easy,Deductive,All roses are flowers. Some flowers fade quick...,No
2,3,Logic,Easy,Pattern,"Which number comes next: 2, 4, 8, 16, ?",32
3,4,Logic,Medium,Conditional,"If all A are B and all B are C, are all A C?",Yes
4,5,Logic,Medium,Deductive,John is taller than Mary. Mary is taller than ...,John


In [24]:
results_df.to_csv("data/qwen_results.csv", index=False)

In [27]:
from google.colab import files

files.download("data/qwen_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
!git config --global user.name "MR-just01"
!git config --global user.email "rawatmaya2908@gmail.com"

In [32]:
!git add data/qwen_results.csv


In [29]:
import os

print(os.getcwd())

/content/open_vs_closed_LLM-


In [30]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   data/qwen_results.csv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	qwen_results.csv

